# 05 — System Integration Benchmark

End-to-end benchmark of the full LLM-driven agent system.

**Tests**:
1. **Step latency waterfall** — DB query → memory read → LLM decision → memory write → tracker log
2. **Scalability** — N = 10, 50, 100, 500 agents
3. **Humanistic scoring** — trajectory patterns after 200 steps

**5 Humanistic Dimensions** (0-1 each):
- Diversity — unique edges / total steps
- Archetype consistency — archetype-appropriate destination choices
- Amenity plausibility — need state matches visited amenity
- Spatial realism — trajectory entropy
- Decision coherence — reasoning consistency across steps

In [ ]:
import asyncio
import sys
import time
import json
import statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import Backend modules
sys.path.insert(0, '..')

sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

In [ ]:
# --- Step latency waterfall (live system benchmark) ---
# Uncomment to run against a running server on localhost:8000
#
# import httpx
# import time
#
# async def bench_step_latency(n_steps=50):
#     timings = []
#     async with httpx.AsyncClient(base_url='http://localhost:8000', timeout=60) as client:
#         # Start simulation
#         await client.post('/api/start')
#         for _ in range(n_steps):
#             t0 = time.perf_counter()
#             resp = await client.post('/api/step')
#             elapsed = (time.perf_counter() - t0) * 1000
#             data = resp.json()
#             timings.append({'step': _, 'latency_ms': elapsed,
#                             'llm_calls': data.get('llm_calls', 0)})
#     return pd.DataFrame(timings)
#
# timings_df = await bench_step_latency()
#

# Placeholder step timings
steps = 50
timings_llm = [max(50, 800 + np.random.randn()*200) for _ in range(steps)]     # LLM-driven
timings_rule = [max(5, 12 + np.random.randn()*3) for _ in range(steps)]         # Rule-based

timing_df = pd.DataFrame({
    'step': list(range(steps)) * 2,
    'latency_ms': timings_llm + timings_rule,
    'mode': ['LLM-driven'] * steps + ['Rule-based'] * steps
})
print('Step latency summary:')
print(timing_df.groupby('mode')['latency_ms'].describe())

In [ ]:
# --- Scalability: N agents ---
# Placeholder data (replace with actual runs)
scale_data = pd.DataFrame({
    'n_agents': [10, 50, 100, 500, 10, 50, 100, 500],
    'step_ms': [120, 380, 730, 3800, 8, 35, 68, 320],
    'mode': ['LLM-driven']*4 + ['Rule-based']*4
})

In [ ]:
# --- Humanistic scoring (placeholder trajectory data) ---
archetypes = ['resident', 'commuter', 'tourist', 'student']

# Simulated scores — replace with actual trajectory analysis
scores_llm = pd.DataFrame([
    {'archetype': a, 'mode': 'LLM-driven',
     'diversity': np.random.beta(6,3),
     'archetype_consistency': np.random.beta(7,2),
     'amenity_plausibility': np.random.beta(6,3),
     'spatial_realism': np.random.beta(5,3),
     'decision_coherence': np.random.beta(6,2)}
    for a in archetypes for _ in range(10)
])
scores_rule = pd.DataFrame([
    {'archetype': a, 'mode': 'Rule-based',
     'diversity': np.random.beta(3,5),
     'archetype_consistency': np.random.beta(3,5),
     'amenity_plausibility': np.random.beta(3,4),
     'spatial_realism': np.random.beta(3,4),
     'decision_coherence': np.random.beta(2,4)}
    for a in archetypes for _ in range(10)
])
scores = pd.concat([scores_llm, scores_rule])
dims = ['diversity', 'archetype_consistency', 'amenity_plausibility', 'spatial_realism', 'decision_coherence']

print(scores.groupby('mode')[dims].mean())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Step latency over time
for mode, grp in timing_df.groupby('mode'):
    axes[0,0].plot(grp['step'], grp['latency_ms'], label=mode, alpha=0.8)
axes[0,0].set_title('Step Latency Over Time (50 steps, placeholder data)')
axes[0,0].set_xlabel('Step')
axes[0,0].set_ylabel('Latency (ms)')
axes[0,0].legend()

# Scalability
for mode, grp in scale_data.groupby('mode'):
    axes[0,1].plot(grp['n_agents'], grp['step_ms'], marker='o', label=mode)
axes[0,1].set_title('Step Latency vs Agent Count')
axes[0,1].set_xlabel('Number of Agents')
axes[0,1].set_ylabel('Step Time (ms)')
axes[0,1].legend()

# Radar chart — humanistic dimensions
from matplotlib.patches import FancyArrowPatch
mean_scores = scores.groupby('mode')[dims].mean()
n = len(dims)
angles = [i / n * 2 * np.pi for i in range(n)] + [0]
ax_radar = plt.subplot(2, 2, 3, polar=True)
for mode_name, row in mean_scores.iterrows():
    vals = row.tolist() + [row.iloc[0]]
    ax_radar.plot(angles, vals, label=mode_name)
    ax_radar.fill(angles, vals, alpha=0.1)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels([d.replace('_', '\n') for d in dims], size=8)
ax_radar.set_title('Humanistic Dimensions Radar')
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# Bar per archetype — archetype consistency
arch_scores = scores.groupby(['archetype', 'mode'])['archetype_consistency'].mean().unstack()
arch_scores.plot(kind='bar', ax=axes[1,1])
axes[1,1].set_title('Archetype Consistency by Agent Type')
axes[1,1].set_ylabel('Score')
axes[1,1].set_ylim(0, 1)
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('results_05_system_benchmark.png', dpi=150)
plt.show()